# Objectif

Créer des variables analytiques et métier.  
**Pertinence dans le mémoire**

Le feature engineering améliore :
- les modèles ML ;
- les analyses décisionnelles ;
- les dashboards.
Ce notebook sert à :

créer les variables métier ;
enrichir les données ;
- préparer les variables pour :
- le Data Warehouse ;
- le Machine Learning ;
- les dashboards ;
- le scoring risque.

Le Feature Engineering est l’une des parties les plus importantes du Data Mining.


In [1]:
#IMPORTATION DES LIBRAIRIES
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("Librairies chargées")

Librairies chargées


In [2]:
#CHARGEMENT DES DONNÉES NETTOYÉES
parc = pd.read_csv(
    "../data/interim/parc_auto_clean.csv"
)

auto = pd.read_csv(
    "../data/interim/sinistre_auto_clean.csv"
)

trc = pd.read_csv(
    "../data/interim/sinistre_trc_clean.csv"
)

print("Datasets nettoyés chargés")

Datasets nettoyés chargés


In [3]:
#APERÇU DES DONNÉES
display(parc.head())

display(auto.head())

display(trc.head())

,immatriculation,type_engin,marque,annee_mise_en_service,age_vehicule_ans,departement_affectation,km_parcourus_estimes,etat_general,assureur
0,PRT-0001-BJ,Pelle hydraulique,Caterpillar,2013,11,Atlantique,202000,Bon,SOBAC
1,PRT-0002-BJ,Bulldozer,Hitachi,2007,17,Collines,307000,Moyen,Allianz Bénin
2,PRT-0003-BJ,Camion benne,Mercedes-Benz,2022,2,Littoral,73000,Bon,SOBAC
3,PRT-0004-BJ,Niveleuse,Volvo CE,2019,5,Zou,76000,Moyen,BGFA Bénin
4,PRT-0005-BJ,Camion citerne,FOTON,2015,9,Atlantique,360000,Moyen,SAAR-Bénin


,id_sinistre,annee,mois,date_accident,immatriculation,type_engin,marque,age_vehicule_ans,etat_vehicule,departement_accident,axe_routier,type_accident,responsabilite,gravite,tiers_implique,assureur,montant_declare_fcfa,franchise_fcfa,montant_indemnise_fcfa,delai_declaration_jours,delai_reglement_jours,statut_dossier,conducteur_anciennete_ans,type_sinistre
0,AUTO-2022-001,2022,7,2022-07-17,PRT-0108-BJ,Niveleuse,Liebherr,13,Mauvais,Ouémé,RN6 Natitingou-Kérou,Collision frontale,Responsable,Matériel grave,Oui,COLINA,1745000,339000,1101000.0,7,119.0,Réglé,17,AUTO
1,AUTO-2022-002,2022,3,2022-03-02,PRT-0068-BJ,Bulldozer,Liebherr,10,Mauvais,Alibori,Voirie urbaine Parakou,Renversement,Responsable,Matériel léger,Oui,SAAR-Bénin,242000,114000,0.0,5,NaN,En cours,19,AUTO
2,AUTO-2022-003,2022,9,2022-09-15,PRT-0318-BJ,Camion benne,MAN,9,Mauvais,Atlantique,RN7 Cotonou-Ouidah,Accrochage mineur,Responsabilité partagée,Matériel léger,Oui,COLINA,471000,270000,0.0,5,NaN,En expertise,10,AUTO
3,AUTO-2022-004,2022,8,2022-08-27,PRT-0586-BJ,Niveleuse,Volvo CE,4,Bon,Borgou,Voirie urbaine Parakou,Accrochage mineur,Responsable,Matériel grave,Non,NSIA Bénin,6000000,321000,4851000.0,2,44.0,Réglé,5,AUTO
4,AUTO-2022-005,2022,9,2022-09-27,PRT-0395-BJ,Camion malaxeur béton,DAF,2,Bon,Ouémé,Piste rurale,Sortie de route,Responsabilité partagée,Matériel léger,Non,COLINA,500000,243000,210000.0,9,87.0,Réglé,17,AUTO


,id_sinistre,annee,mois,date_declaration,code_chantier,nom_chantier,type_travaux,departement,phase_chantier,nature_sinistre,tiers_implique,engin_implique,assureur,montant_declare_fcfa,franchise_fcfa,montant_indemnise_fcfa,delai_declaration_jours,delai_reglement_jours,statut_dossier,responsabilite,type_sinistre
0,TRC-2022-001,2022,4,2022-04-13,CH-003,Route Parakou-N'Dali,Terrassement,Borgou,Gros œuvre,Conduite eau endommagée,SONEB,Pelle hydraulique,NSIA Bénin,1858000,1016000,0.0,8,NaN,En expertise,PORTEO entière,TRC
1,TRC-2022-002,2022,10,2022-10-19,CH-029,Route Covè-Zagnanado,Terrassement,Zou,Gros œuvre,Dommages propriété riveraine,Riverain particulier,Camion benne,UAB,2780000,1208000,0.0,21,NaN,En cours,PORTEO entière,TRC
2,TRC-2022-003,2022,8,2022-08-07,CH-007,Bitumage Natitingou-Tanguiéta,Bitumage,Atacora,Terrassement,Dommages propriété riveraine,Riverain particulier,Chargeur,NSIA Bénin,4284000,984000,2571000.0,20,156.0,Litigieux,PORTEO partielle,TRC
3,TRC-2022-004,2022,1,2022-01-17,CH-013,Pont sur Ouémé à Adjohoun,Ouvrage d'art,Ouémé,Finition,Effondrement de talus,Collectivité locale,Chargeur,NSIA Bénin,17294000,375000,16101000.0,11,45.0,Litigieux,PORTEO partielle,TRC
4,TRC-2022-005,2022,10,2022-10-08,CH-014,Voirie Parakou Zone industrielle,VRD,Borgou,Terrassement,Dommages propriété riveraine,Riverain particulier,Niveleuse,UAB,2637000,682000,1562000.0,6,184.0,Réglé,PORTEO entière,TRC


In [4]:
# FUSION PARC + SINISTRE AUTO
auto_merge = auto.merge(

    parc,

    on="immatriculation",

    how="left",

    suffixes=("_sinistre", "_parc")

)

print("Fusion effectuée")

Fusion effectuée


In [5]:
# DIMENSIONS APRÈS FUSION
print(auto_merge.shape)

display(auto_merge.head())

(410, 32)


,id_sinistre,annee,mois,date_accident,immatriculation,type_engin_sinistre,marque_sinistre,age_vehicule_ans_sinistre,etat_vehicule,departement_accident,axe_routier,type_accident,responsabilite,gravite,tiers_implique,assureur_sinistre,montant_declare_fcfa,franchise_fcfa,montant_indemnise_fcfa,delai_declaration_jours,delai_reglement_jours,statut_dossier,conducteur_anciennete_ans,type_sinistre,type_engin_parc,marque_parc,annee_mise_en_service,age_vehicule_ans_parc,departement_affectation,km_parcourus_estimes,etat_general,assureur_parc
0,AUTO-2022-001,2022,7,2022-07-17,PRT-0108-BJ,Niveleuse,Liebherr,13,Mauvais,Ouémé,RN6 Natitingou-Kérou,Collision frontale,Responsable,Matériel grave,Oui,COLINA,1745000,339000,1101000.0,7,119.0,Réglé,17,AUTO,Niveleuse,Liebherr,2011,13,Atlantique,504000,Mauvais,COLINA
1,AUTO-2022-002,2022,3,2022-03-02,PRT-0068-BJ,Bulldozer,Liebherr,10,Mauvais,Alibori,Voirie urbaine Parakou,Renversement,Responsable,Matériel léger,Oui,SAAR-Bénin,242000,114000,0.0,5,NaN,En cours,19,AUTO,Bulldozer,Liebherr,2014,10,Ouémé,250000,Mauvais,SAAR-Bénin
2,AUTO-2022-003,2022,9,2022-09-15,PRT-0318-BJ,Camion benne,MAN,9,Mauvais,Atlantique,RN7 Cotonou-Ouidah,Accrochage mineur,Responsabilité partagée,Matériel léger,Oui,COLINA,471000,270000,0.0,5,NaN,En expertise,10,AUTO,Camion benne,MAN,2015,9,Atlantique,295000,Mauvais,COLINA
3,AUTO-2022-004,2022,8,2022-08-27,PRT-0586-BJ,Niveleuse,Volvo CE,4,Bon,Borgou,Voirie urbaine Parakou,Accrochage mineur,Responsable,Matériel grave,Non,NSIA Bénin,6000000,321000,4851000.0,2,44.0,Réglé,5,AUTO,Niveleuse,Volvo CE,2020,4,Atlantique,148000,Bon,NSIA Bénin
4,AUTO-2022-005,2022,9,2022-09-27,PRT-0395-BJ,Camion malaxeur béton,DAF,2,Bon,Ouémé,Piste rurale,Sortie de route,Responsabilité partagée,Matériel léger,Non,COLINA,500000,243000,210000.0,9,87.0,Réglé,17,AUTO,Camion malaxeur béton,DAF,2022,2,Littoral,80000,Bon,COLINA


## 1. Création du coût total

```python
cout_total = montant_indemnise_fcfa + franchise_fcfa
```

In [6]:
# CRÉATION DU COÛT TOTAL
auto_merge["cout_total_fcfa"] = (

    auto_merge["montant_indemnise_fcfa"]

    +

    auto_merge["franchise_fcfa"]

)

trc["cout_total_fcfa"] = (

    trc["montant_indemnise_fcfa"]

    +

    trc["franchise_fcfa"]

)

print("Variable coût total créée")

Variable coût total créée


In [7]:
# VÉRIFICATION COÛT TOTAL
display(

    auto_merge[
        [
            "montant_indemnise_fcfa",
            "franchise_fcfa",
            "cout_total_fcfa"
        ]
    ].head()

)

,montant_indemnise_fcfa,franchise_fcfa,cout_total_fcfa
0,1101000.0,339000,1440000.0
1,0.0,114000,114000.0
2,0.0,270000,270000.0
3,4851000.0,321000,5172000.0
4,210000.0,243000,453000.0


## 2. Création de classe_age

- Récent ;
- Moyen ;
- Ancien.

In [9]:
# CRÉATION DE CLASSE_AGE
def classe_age(age):

    if age <= 3:
        return "RECENT"

    elif age <= 7:
        return "MOYEN"

    else:
        return "ANCIEN"

In [8]:
# APPLICATION CLASSE_AGE
auto_merge["classe_age"] = auto_merge[
    "age_vehicule_ans_parc"
].apply(classe_age)

print("classe_age créée")

classe_age créée


## 3. Création categorie_km

- Faible ;
- Moyen ;
- Élevé.

In [10]:
#CRÉATION CATEGORIE_KM
def categorie_km(km):

    if km < 50000:
        return "FAIBLE"

    elif km < 150000:
        return "MOYEN"

    else:
        return "ELEVE"

In [11]:
# APPLICATION CATEGORIE_KM
auto_merge["categorie_km"] = auto_merge[
    "km_parcourus_estimes"
].apply(categorie_km)

print("categorie_km créée")

categorie_km créée


## 4. Création niveau_experience

- Débutant ;
- Intermédiaire ;
- Expérimenté.

In [12]:
# CRÉATION NIVEAU_EXPERIENCE
def niveau_experience(exp):

    if exp <= 2:
        return "DEBUTANT"

    elif exp <= 5:
        return "INTERMEDIAIRE"

    else:
        return "EXPERIMENTE"

In [13]:
# APPLICATION NIVEAU_EXPERIENCE
auto_merge["niveau_experience"] = auto_merge[
    "conducteur_anciennete_ans"
].apply(niveau_experience)

print("niveau_experience créée")

niveau_experience créée


## 5. Création de variables cibles

- sinistre grave ;
- niveau risque ;
- coût futur.

In [14]:
# CRÉATION SCORE DÉLAI
auto_merge["score_delai"] = (

    auto_merge["delai_declaration_jours"]

    +

    auto_merge["delai_reglement_jours"]

)

In [15]:
# CRÉATION NIVEAU RISQUE
def niveau_risque(cout):

    if cout < 1000000:
        return "FAIBLE"

    elif cout < 5000000:
        return "MOYEN"

    else:
        return "ELEVE"

In [16]:
# APPLICATION NIVEAU RISQUE
auto_merge["niveau_risque"] = auto_merge[
    "cout_total_fcfa"
].apply(niveau_risque)

trc["niveau_risque"] = trc[
    "cout_total_fcfa"
].apply(niveau_risque)

print("niveau_risque créé")

niveau_risque créé


In [17]:
# CRÉATION VARIABLE CIBLE SINISTRE_GRAVE
auto_merge["sinistre_grave"] = np.where(

    auto_merge["gravite"].isin(
        [
            "GRAVE",
            "MORTEL",
            "CORPOREL"
        ]
    ),

    1,

    0
)

print("Variable cible créée")

Variable cible créée


In [18]:
# FRÉQUENCE DES SINISTRES PAR VÉHICULE
frequence = auto_merge.groupby(
    "immatriculation"
).size().reset_index()

frequence.columns = [
    "immatriculation",
    "frequence_sinistre"
]

display(frequence.head())

,immatriculation,frequence_sinistre
0,PRT-0001-BJ,1
1,PRT-0003-BJ,1
2,PRT-0004-BJ,1
3,PRT-0006-BJ,1
4,PRT-0007-BJ,1


In [19]:
# AJOUT FRÉQUENCE AU DATASET
auto_merge = auto_merge.merge(

    frequence,

    on="immatriculation",

    how="left"

)

print("Fréquence ajoutée")

Fréquence ajoutée


In [19]:
# SCORE RISQUE PRÉLIMINAIRE
auto_merge["score_risque"] = (

    auto_merge["frequence_sinistre"] * 0.4

    +

    auto_merge["age_vehicule_ans_parc"] * 0.2

    +

    auto_merge["score_delai"] * 0.1

)

In [20]:
# NORMALISATION SCORE RISQUE
score_min = auto_merge["score_risque"].min()

score_max = auto_merge["score_risque"].max()

auto_merge["score_risque"] = (

    (
        auto_merge["score_risque"] - score_min
    )

    /

    (
        score_max - score_min
    )

) * 100

KeyError: 'score_risque'

In [ ]:
# ARRONDIR SCORE RISQUE
auto_merge["score_risque"] = auto_merge[
    "score_risque"
].round(2)

display(
    auto_merge[
        [
            "immatriculation",
            "score_risque"
        ]
    ].head()
)

In [ ]:
# CRÉATION TRANCHE COÛT
def tranche_cout(cout):

    if cout < 1000000:
        return "FAIBLE"

    elif cout < 5000000:
        return "MOYEN"

    else:
        return "ELEVE"

In [23]:
# APPLICATION TRANCHE COÛT
auto_merge["tranche_cout"] = auto_merge[
    "cout_total_fcfa"
].apply(tranche_cout)

trc["tranche_cout"] = trc[
    "cout_total_fcfa"
].apply(tranche_cout)

print("tranche_cout créée")

tranche_cout créée


In [21]:
# EXTRACTION VARIABLES TEMPORELLES
auto_merge["date_accident"] = pd.to_datetime(
    auto_merge["date_accident"]
)

auto_merge["annee_accident"] = auto_merge[
    "date_accident"
].dt.year

auto_merge["mois_accident"] = auto_merge[
    "date_accident"
].dt.month

auto_merge["jour_semaine"] = auto_merge[
    "date_accident"
].dt.day_name()

print("Variables temporelles créées")

Variables temporelles créées


In [22]:
# CRÉATION WEEKEND
auto_merge["est_weekend"] = auto_merge[
    "jour_semaine"
].isin(
    [
        "Saturday",
        "Sunday"
    ]
)

In [23]:
# ANALYSE DES VARIABLES CRÉÉES
display(

    auto_merge[
        [
            "classe_age",
            "categorie_km",
            "niveau_experience",
            "niveau_risque",
            "tranche_cout"
        ]
    ].head()

)

KeyError: "['tranche_cout'] not in index"

In [27]:
# VÉRIFICATION DES NULL APRÈS FEATURE
display(
    auto_merge.isnull().sum()
)

id_sinistre                  0
annee                        0
mois                         0
date_accident                0
immatriculation              0
type_engin_sinistre          0
marque_sinistre              0
age_vehicule_ans_sinistre    0
etat_vehicule                0
departement_accident         0
axe_routier                  0
type_accident                0
responsabilite               0
gravite                      0
tiers_implique               0
assureur_sinistre            0
montant_declare_fcfa         0
franchise_fcfa               0
montant_indemnise_fcfa       0
delai_declaration_jours      0
delai_reglement_jours        0
statut_dossier               0
conducteur_anciennete_ans    0
type_sinistre                0
type_engin_parc              0
marque_parc                  0
annee_mise_en_service        0
age_vehicule_ans_parc        0
departement_affectation      0
km_parcourus_estimes         0
etat_general                 0
assureur_parc                0
cout_tot

In [42]:
# SÉLECTION VARIABLES MACHINE LEARNING
colonnes_ml = [

    "age_vehicule_ans_parc",
    "km_parcourus_estimes",
    "conducteur_anciennete_ans",
    "frequence_sinistre",
    "score_delai",
    "cout_total_fcfa",
    "score_risque",
    "sinistre_grave"

]

dataset_ml = auto_merge[colonnes_ml]

display(dataset_ml.head())

,age_vehicule_ans_parc,km_parcourus_estimes,conducteur_anciennete_ans,frequence_sinistre,score_delai,cout_total_fcfa,score_risque,sinistre_grave
0,13,504000,17,1,126.0,1440000.0,93.46,0
1,10,250000,19,1,5.0,114000.0,10.46,0
2,9,295000,10,2,5.0,270000.0,11.76,0
3,4,148000,5,3,46.0,5172000.0,34.64,0
4,2,80000,17,2,96.0,453000.0,62.09,0


In [40]:
# EXPORT DATASET ANALYTIQUE
auto_merge.to_csv(

    "../data/processed/dataset_analytique.csv",

    index=False

)

print("Dataset analytique exporté")

Dataset analytique exporté


In [43]:
# EXPORT DATASET MACHINE LEARNING
dataset_ml.to_csv(

    "../data/processed/dataset_ml.csv",

    index=False

)

print("Dataset ML exporté")

Dataset ML exporté


In [31]:
# EXPORT TRC ENRICHI
trc.to_csv(

    "../data/processed/sinistre_trc_enrichi.csv",

    index=False

)

print("TRC enrichi exporté")

TRC enrichi exporté


In [45]:
# RAPPORT FEATURE ENGINEERING
rapport_features = pd.DataFrame({

    "Variables_creees": [

        "cout_total_fcfa",
        "classe_age",
        "categorie_km",
        "niveau_experience",
        "niveau_risque",
        "score_risque",
        "tranche_cout",
        "sinistre_grave"

    ]

})

rapport_features.to_csv(

    "../reports/table/rapport_features.csv",

    index=False

)

print("Rapport feature engineering exporté")

Rapport feature engineering exporté


# Synthèse du Feature Engineering

## Variables créées

### Variables métier
- classe_age
- categorie_km
- niveau_experience

### Variables analytiques
- cout_total_fcfa
- score_delai
- frequence_sinistre

### Variables Machine Learning
- score_risque
- niveau_risque
- sinistre_grave

## Résultat

Les données sont maintenant enrichies et prêtes pour :
- le Data Warehouse ;
- le Machine Learning ;
- le scoring risque ;
- les dashboards Power BI.